## Path configuration

In [21]:
from pathlib import Path
import os
import sys

PROJECT_NAME = "MALDIAlign"
cwd = Path().resolve()

target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

if target is not None and target != cwd:
    os.chdir(target)
    sys.path.append(str(target))

## Imports

In [22]:
import pandas as pd
from pathlib import Path

## Results

In [23]:
OUTPUT_DIR = Path("/export/usuarios01/agnavarr/MALDIAlign/experiments/novelty_detection/combined/20260626_120116")
df_unseen = pd.read_csv(OUTPUT_DIR / "unseen_species_ood_by_criterion.csv")
df_gmm_p05 = df_unseen[df_unseen["percentile"] == 0.5][["species", "n_total", "GMM_n_ood", "GMM_n_indist", "GMM_pct_ood"]]
df_gmm_p05 = df_gmm_p05.sort_values("GMM_pct_ood")

In [24]:
TARGET_SPECIES = [
    "Klebsiella_Pneumoniae", "Escherichia_Coli", "Staphylococcus_Aureus",
    "Pseudomonas_Aeruginosa", "Enterococcus_Faecium", "Enterobacter_cloacae_complex",
]
train_genera = set(sp.split("_")[0] for sp in TARGET_SPECIES)

# Extract the genus of each unseen species
df_gmm_p05["genus"] = df_gmm_p05["species"].str.split("_").str[0]

# Mark whether the genus matches any training genus
df_gmm_p05["same_genus_as_train"] = df_gmm_p05["genus"].isin(train_genera)

# Split into two tables
df_same_genus = df_gmm_p05[df_gmm_p05["same_genus_as_train"]].sort_values("GMM_pct_ood")
df_diff_genus = df_gmm_p05[~df_gmm_p05["same_genus_as_train"]].sort_values("GMM_pct_ood")

print(f"\n===== SAME GENUS AS TRAINING SET (n={len(df_same_genus)}) =====")
print(df_same_genus.to_string(index=False))

print(f"\n===== DIFFERENT GENUS FROM TRAINING SET (n={len(df_diff_genus)}) =====")
print(df_diff_genus.to_string(index=False))


===== SAME GENUS AS TRAINING SET (n=48) =====
                        species  n_total  GMM_n_ood  GMM_n_indist  GMM_pct_ood          genus  same_genus_as_train
           Klebsiella_Variicola      664          2           662          0.3     Klebsiella                 True
       Enterobacter_Bugandensis       48          1            47          2.1   Enterobacter                 True
         Enterobacter_Aerogenes      433         17           416          3.9   Enterobacter                 True
       Klebsiella_Michiganensis       30          4            26         13.3     Klebsiella                 True
           Klebsiella_Aerogenes      678        180           498         26.5     Klebsiella                 True
             Enterococcus_Hirae       73         31            42         42.5   Enterococcus                 True
      Pseudomonas_Citronellolis       14          6             8         42.9    Pseudomonas                 True
            Enterococcus_Durans  

In [25]:
# Detection rate bins across all 336 species
bins = [
    (99, 100.001, ">=99%"),
    (95, 99,      "95-99%"),
    (80, 95,      "80-95%"),
    (50, 80,      "50-80%"),
    (10, 50,      "10-50%"),
    (0, 10,       "<10%"),
]

print(f"\n===== DETECTION RATE BINS (n={len(df_gmm_p05)} species) =====")
rows = []
for lo, hi, label in bins:
    mask = (df_gmm_p05["GMM_pct_ood"] >= lo) & (df_gmm_p05["GMM_pct_ood"] < hi)
    count = mask.sum()
    rows.append((label, count))
    print(f"{label:>10}: {count}")

print("\nSpecies with detection rate <10%:")
print(
    df_gmm_p05[df_gmm_p05["GMM_pct_ood"] < 10][
        ["species", "genus", "n_total", "GMM_pct_ood"]
    ].to_string(index=False)
)


===== DETECTION RATE BINS (n=336 species) =====
     >=99%: 251
    95-99%: 30
    80-95%: 29
    50-80%: 16
    10-50%: 6
      <10%: 4

Species with detection rate <10%:
                 species        genus  n_total  GMM_pct_ood
    Klebsiella_Variicola   Klebsiella      664          0.3
Enterobacter_Bugandensis Enterobacter       48          2.1
  Enterobacter_Aerogenes Enterobacter      433          3.9
         Shigella_Sonnei     Shigella       12          8.3


In [26]:
# Show all 336 species together, sorted, without distinguishing by genus
df_all_sorted = df_gmm_p05.sort_values("GMM_pct_ood").reset_index(drop=True)
print(f"\n===== ALL SORTED SPECIES (n={len(df_all_sorted)}) =====")
print(df_all_sorted[["species", "n_total", "GMM_n_ood", "GMM_pct_ood"]].to_string(index=False))


===== ALL SORTED SPECIES (n=336) =====
                                  species  n_total  GMM_n_ood  GMM_pct_ood
                     Klebsiella_Variicola      664          2          0.3
                 Enterobacter_Bugandensis       48          1          2.1
                   Enterobacter_Aerogenes      433         17          3.9
                          Shigella_Sonnei       12          1          8.3
                 Klebsiella_Michiganensis       30          4         13.3
                     Klebsiella_Aerogenes      678        180         26.5
                       Citrobacter_Koseri     1767        691         39.1
                       Enterococcus_Hirae       73         31         42.5
                Pseudomonas_Citronellolis       14          6         42.9
                      Enterococcus_Durans       26         12         46.2
               Staphylococcus_Lugdunensis      950        487         51.3
                      Enterococcus_Dispar       35         1